# FractalTrainer — Sprint 19b: Omniglot 5-way K-shot head-to-head

The paper cites MAML (Finn 2017) and ProtoNets (Snell 2017) as the closest meta-learning analogs but never runs a head-to-head. This notebook fixes that on **Omniglot 5-way 5-shot**, the canonical few-shot-learning benchmark.

### Methods compared

All three use the same `784 → 64 → 32` MLP backbone for fairness — the only thing that differs is what happens on top of the 32-d embedding.

| Method | How the 5-way classification is produced |
|---|---|
| **pixel-kNN** (naive baseline) | 1-NN in raw 784-d pixel space against 25 support examples |
| **MLP-ProtoNets** (Snell 2017) | Meta-trained 784→64→32 encoder. Per-class prototype = mean of support embeddings. Classify query by L2 distance to prototypes. |
| **FractalTrainer spawn+context** | Registry seeded from background episodes. For each test episode, spawn a new 5-way MLP on the 25 support samples with K=3 nearest-neighbor context injection. |

### Eval setup

- Omniglot: 1623 character classes across 50 alphabets, 20 examples per character
- Standard split: 1200 background (train) + 423 evaluation (test) characters
- 5-way 5-shot episodes: 5 novel characters × (5 support + 15 query) = 100 samples per episode
- **Eval metric**: mean query accuracy across 100 test episodes, ± std, + paired Welch t-test between methods

### Pre-registered outcomes

- **If FractalTrainer ties or beats ProtoNets** (within ±3 pp): major positive for the paper — registry+context is a real alternative to meta-learning.
- **If FractalTrainer loses by > 3 pp to ProtoNets**: also a useful finding — paper honestly states "registry-based spawning underperforms meta-learning by X pp on few-shot."
- **Both methods beat pixel-kNN by > 20 pp**: sanity check. If either fails this, the backbone is too weak.

### Cost & runtime (T4 GPU)

| Mode | Seed episodes | Test episodes | Total wall time |
|---|---|---|---|
| `SMOKE=True` | 20 | 10 | ~3 min |
| `SMOKE=False` | 200 | 100 | ~30 min |

### Results persistence

Triple backup: `/content/` (ephemeral), Google Drive (if `USE_DRIVE=True`), inline JSON in Cell 12 (survives via notebook save).


## Cell 1. Config — EDIT HERE

In [ ]:
SMOKE = True
USE_DRIVE = True
DRIVE_SAVE_DIR = "FractalTrainer_results"

N_WAY = 5
K_SHOT = 5
N_QUERY = 15
CONTEXT_K = 3           # how many neighbors to pull context from on spawn
SPAWN_STEPS = 200       # per-expert training steps at episode spawn time
PROTONET_ITERS = 3000   # meta-training iterations for ProtoNets
PROTONET_LR = 1e-3
PROBE_SIZE = 100        # fixed probe batch for signature computation

if SMOKE:
    N_SEED_EPISODES = 20
    N_TEST_EPISODES = 10
    PROTONET_ITERS = 500
else:
    N_SEED_EPISODES = 200
    N_TEST_EPISODES = 100

print(f"mode={'SMOKE' if SMOKE else 'FULL'}  seed_eps={N_SEED_EPISODES}  test_eps={N_TEST_EPISODES}  "
      f"protonet_iters={PROTONET_ITERS}")

## Cell 2. Clone + device

In [ ]:
import os, sys, subprocess, time
REPO_URL = "https://github.com/vegarjr/FractalTrainer.git"
REPO_DIR = "/content/FractalTrainer"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
head = subprocess.check_output(["git", "-C", REPO_DIR, "log", "--oneline", "-1"]).decode().strip()
print("repo HEAD:", head)

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
DRIVE_PATH = None
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_PATH = os.path.join("/content/drive/MyDrive", DRIVE_SAVE_DIR)
        os.makedirs(DRIVE_PATH, exist_ok=True)
        print("Drive mounted:", DRIVE_PATH)
    except Exception as e:
        print("Drive not available:", e)

## Cell 3. Load Omniglot, build background / evaluation splits

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms

# 28x28 grayscale, flattened to 784-d later. Omniglot default is 105x105 — resize.
omni_transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),           # [0, 1]
    transforms.Lambda(lambda x: 1.0 - x),  # invert: char = 1, background = 0 (closer to MNIST)
])

omni_bg = datasets.Omniglot(root="/content/omniglot", background=True,  download=True, transform=omni_transform)
omni_ev = datasets.Omniglot(root="/content/omniglot", background=False, download=True, transform=omni_transform)

# Each Omniglot item has a (image, integer class idx) pair. Build class → list[idx] maps.
def _class_index(ds):
    idx_by_class = {}
    for i, (_, y) in enumerate(ds._flat_character_images):
        idx_by_class.setdefault(y, []).append(i)
    return idx_by_class

bg_idx_by_class = _class_index(omni_bg)
ev_idx_by_class = _class_index(omni_ev)
print(f"background: {len(bg_idx_by_class)} classes, ~{len(next(iter(bg_idx_by_class.values())))} examples/class")
print(f"evaluation: {len(ev_idx_by_class)} classes, ~{len(next(iter(ev_idx_by_class.values())))} examples/class")

# Fixed probe batch — 100 random background images, held away from seed/test episodes
probe_rng = np.random.default_rng(0)
# Sample PROBE_SIZE images from background, uniform across classes
bg_classes = sorted(bg_idx_by_class.keys())
probe_items = []
for _ in range(PROBE_SIZE):
    c = probe_rng.choice(bg_classes)
    i = probe_rng.choice(bg_idx_by_class[c])
    probe_items.append(i)
probe_items_set = set(probe_items)
probe_X = torch.stack([omni_bg[i][0] for i in probe_items]).view(PROBE_SIZE, -1).to(DEVICE)
print(f"probe shape: {probe_X.shape}  device: {probe_X.device}")

# Signature dimension = PROBE_SIZE × N_WAY (softmax on 5-way output)
SIG_DIM = PROBE_SIZE * N_WAY
print(f"signature dim: {SIG_DIM}")

## Cell 4. Episode sampler

In [ ]:
def sample_episode(ds, idx_by_class, rng, n_way=N_WAY, k_shot=K_SHOT,
                   n_query=N_QUERY, exclude=None):
    """Sample a single 5-way K-shot episode.

    Returns:
        support_x: (n_way*k_shot, 784) tensor, episode-label 0..n_way-1 ordered
        support_y: (n_way*k_shot,) tensor of episode labels 0..n_way-1
        query_x:   (n_way*n_query, 784)
        query_y:   (n_way*n_query,)
        chars:     tuple of n_way original class indices selected
    """
    classes = sorted(idx_by_class.keys())
    exclude = set(exclude or ())
    picks = []
    while len(picks) < n_way:
        c = int(rng.choice(classes))
        if c in picks or c in exclude: continue
        if len(idx_by_class[c]) < k_shot + n_query: continue
        picks.append(c)
    sup_x, sup_y, qry_x, qry_y = [], [], [], []
    for lab, c in enumerate(picks):
        idx = list(idx_by_class[c])
        rng.shuffle(idx)
        for j in idx[:k_shot]:
            sup_x.append(ds[j][0].view(-1)); sup_y.append(lab)
        for j in idx[k_shot:k_shot + n_query]:
            qry_x.append(ds[j][0].view(-1)); qry_y.append(lab)
    return (torch.stack(sup_x).to(DEVICE),
            torch.tensor(sup_y, dtype=torch.long, device=DEVICE),
            torch.stack(qry_x).to(DEVICE),
            torch.tensor(qry_y, dtype=torch.long, device=DEVICE),
            tuple(picks))

# Sanity
rng = np.random.default_rng(0)
sx, sy, qx, qy, chars = sample_episode(omni_bg, bg_idx_by_class, rng)
print(f"support: {sx.shape} {sy.shape}   query: {qx.shape} {qy.shape}   chars={chars}")

## Cell 5. Baseline: pixel-kNN

In [ ]:
def pixel_knn_eval(support_x, support_y, query_x, query_y, k=1):
    # pairwise L2 distances (query × support)
    d = torch.cdist(query_x, support_x)  # (Q, S)
    topk = d.topk(k, dim=1, largest=False).indices  # (Q, k)
    votes = support_y[topk]  # (Q, k)
    if k == 1:
        pred = votes.squeeze(-1)
    else:
        pred = torch.mode(votes, dim=1).values
    return float((pred == query_y).float().mean().item())

rng = np.random.default_rng(12345)
knn_accs = []
for _ in range(min(20, N_TEST_EPISODES * 2)):  # quick calibration of baseline
    sx, sy, qx, qy, _ = sample_episode(omni_ev, ev_idx_by_class, rng)
    knn_accs.append(pixel_knn_eval(sx, sy, qx, qy))
print(f"pixel-kNN quick-check (n=20): {np.mean(knn_accs):.3f} ± {np.std(knn_accs):.3f}")

## Cell 6. MLP-ProtoNets (meta-train on background)

In [ ]:
import torch.nn as nn

class MLPEncoder(nn.Module):
    """Matches ContextAwareMLP's 784→64→32 embedding spine."""
    def __init__(self, in_dim=784, hid=64, out_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hid)
        self.fc2 = nn.Linear(hid, out_dim)
    def forward(self, x):
        h = torch.relu(self.fc1(x))
        return torch.relu(self.fc2(h))  # 32-d embedding

def protonet_episode_loss(encoder, sx, sy, qx, qy, n_way=N_WAY):
    sup_emb = encoder(sx)    # (S, 32)
    qry_emb = encoder(qx)    # (Q, 32)
    protos = torch.stack([sup_emb[sy == k].mean(0) for k in range(n_way)])  # (n_way, 32)
    d = torch.cdist(qry_emb, protos)     # (Q, n_way)
    logits = -d
    loss = F.cross_entropy(logits, qy)
    acc = float((logits.argmax(1) == qy).float().mean().item())
    return loss, acc

torch.manual_seed(0)
enc = MLPEncoder().to(DEVICE)
opt = torch.optim.Adam(enc.parameters(), lr=PROTONET_LR)
rng_mt = np.random.default_rng(7)
t0 = time.time()
for it in range(PROTONET_ITERS):
    sx, sy, qx, qy, _ = sample_episode(omni_bg, bg_idx_by_class, rng_mt)
    loss, acc = protonet_episode_loss(enc, sx, sy, qx, qy)
    opt.zero_grad(); loss.backward(); opt.step()
    if (it + 1) % max(1, PROTONET_ITERS // 10) == 0:
        print(f"  protonet iter {it+1:>4d}/{PROTONET_ITERS}  loss={loss.item():.4f}  train_acc={acc:.3f}  [{time.time()-t0:.0f}s]")
print(f"ProtoNets meta-training done in {time.time()-t0:.0f}s")

## Cell 7. FractalTrainer spawn+context setup

In [ ]:
from fractaltrainer.integration.context_mlp import ContextAwareMLP
from fractaltrainer.integration.signatures import softmax_signature
from fractaltrainer.integration.context_injection import gather_context, ContextSpec
from fractaltrainer.registry import FractalEntry, FractalRegistry


def train_episode_expert(sx, sy, n_way=N_WAY, n_steps=SPAWN_STEPS, lr=0.01,
                          batch_size=25, neighbors=None, neighbor_distances=None,
                          context_scale=1.0, seed=0):
    """Train a fresh 5-way ContextAwareMLP on the 25 support examples,
    optionally with context injection from `neighbors`.
    Returns: (model, signature np.ndarray of length SIG_DIM)."""
    torch.manual_seed(seed); np.random.seed(seed)
    cs = float(context_scale) if neighbors else 0.0
    model = ContextAwareMLP(n_classes=n_way, context_scale=cs).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    # Small dataset: 25 support examples. Mini-batch = all.
    for step in range(n_steps):
        # Shuffle each "epoch" (use indices)
        perm = torch.randperm(sx.size(0), device=DEVICE)
        xb, yb = sx[perm], sy[perm]
        if neighbors is not None and cs > 0.0:
            ctx = gather_context(neighbors, xb, ContextSpec(), distances=neighbor_distances)
        else:
            ctx = None
        opt.zero_grad()
        logits = model(xb, context=ctx)
        loss = F.cross_entropy(logits, yb)
        loss.backward(); opt.step()
    model.eval()
    # Signature on probe (no context at signature time — invariant per paper)
    sig = softmax_signature(model, probe_X)
    return model, sig


def eval_episode_query(model, qx, qy, neighbors=None, neighbor_distances=None,
                       context_scale=1.0):
    """Query accuracy. If neighbors provided, apply context at inference too
    (paper §2 'eval-time context fix')."""
    model.eval()
    with torch.no_grad():
        if neighbors is not None and context_scale > 0.0:
            ctx = gather_context(neighbors, qx, ContextSpec(), distances=neighbor_distances)
        else:
            ctx = None
        logits = model(qx, context=ctx)
        pred = logits.argmax(dim=1)
    return float((pred == qy).float().mean().item())

## Cell 8. Seed the FractalTrainer registry from background episodes

In [ ]:
registry = FractalRegistry()
seed_models = {}  # entry_name → trained model, for context access at test time

rng_seed = np.random.default_rng(11)
t0 = time.time()
for ep in range(N_SEED_EPISODES):
    sx, sy, _, _, chars = sample_episode(omni_bg, bg_idx_by_class, rng_seed)
    # Context from current registry: top-K by signature distance of a *probe* look-ahead
    if len(registry) > 0:
        # Use a probe-based tentative signature to find neighbors: train a tiny no-context pass
        # and get its sig, then look up. Simpler: seed episodes spawn with NO context (cold).
        neighbors, ndist = None, None
    else:
        neighbors, ndist = None, None
    model, sig = train_episode_expert(sx, sy, seed=ep,
                                       neighbors=neighbors,
                                       neighbor_distances=ndist,
                                       context_scale=0.0)  # seed episodes: no context (paper Arm A)
    name = f"seed_ep{ep:04d}"
    entry = FractalEntry(name=name, signature=sig,
                         metadata={"task": "_".join(str(c) for c in chars),
                                    "chars": list(chars), "ep": ep})
    registry.add(entry)
    seed_models[name] = model
    if (ep + 1) % max(1, N_SEED_EPISODES // 10) == 0:
        print(f"  seeded ep {ep+1:>4d}/{N_SEED_EPISODES}  reg_size={len(registry)}  [{time.time()-t0:.0f}s]")
print(f"\nRegistry seeded: {len(registry)} entries in {time.time()-t0:.0f}s")

## Cell 9. Evaluate all three methods on test episodes

In [ ]:
# ProtoNets eval helper
def protonet_eval(enc, sx, sy, qx, qy, n_way=N_WAY):
    enc.eval()
    with torch.no_grad():
        sup_emb = enc(sx); qry_emb = enc(qx)
        protos = torch.stack([sup_emb[sy == k].mean(0) for k in range(n_way)])
        d = torch.cdist(qry_emb, protos)
        pred = (-d).argmax(1)
    return float((pred == qy).float().mean().item())

rng_test = np.random.default_rng(2024)
accs = {"pixel_knn": [], "protonets": [], "fractal": [], "fractal_no_ctx": []}
t_total = time.time()
for ep in range(N_TEST_EPISODES):
    sx, sy, qx, qy, chars = sample_episode(omni_ev, ev_idx_by_class, rng_test)
    # 1) pixel-kNN
    accs["pixel_knn"].append(pixel_knn_eval(sx, sy, qx, qy))
    # 2) ProtoNets (zero-shot meta-trained)
    accs["protonets"].append(protonet_eval(enc, sx, sy, qx, qy))
    # 3) FractalTrainer spawn+context: find K nearest neighbors via a tentative sig
    #    Strategy: first train a cold-start expert, use its signature to find neighbors,
    #    then retrain with context from those neighbors.
    cold_model, cold_sig = train_episode_expert(sx, sy, seed=ep + 10000,
                                                  neighbors=None, context_scale=0.0)
    res = registry.find_nearest(cold_sig, k=CONTEXT_K, query_name=f"test_ep{ep}")
    neighbors = [seed_models[e.name] for e in res.entries]
    neighbor_distances = list(res.distances)
    # Arm B: retrain with context
    warm_model, _ = train_episode_expert(sx, sy, seed=ep + 10000,
                                          neighbors=neighbors,
                                          neighbor_distances=neighbor_distances,
                                          context_scale=1.0)
    accs["fractal"].append(eval_episode_query(warm_model, qx, qy,
                                                neighbors=neighbors,
                                                neighbor_distances=neighbor_distances,
                                                context_scale=1.0))
    # Arm A: the cold model evaluated on query (no context — baseline)
    accs["fractal_no_ctx"].append(eval_episode_query(cold_model, qx, qy,
                                                      neighbors=None, context_scale=0.0))
    if (ep + 1) % max(1, N_TEST_EPISODES // 10) == 0:
        print(f"  ep {ep+1:>3d}/{N_TEST_EPISODES}  "
              f"kNN={np.mean(accs['pixel_knn']):.3f}  "
              f"proto={np.mean(accs['protonets']):.3f}  "
              f"fractal={np.mean(accs['fractal']):.3f}  "
              f"fractal_noctx={np.mean(accs['fractal_no_ctx']):.3f}  "
              f"[{time.time()-t_total:.0f}s]")
print(f"\nEval done in {time.time()-t_total:.0f}s")

## Cell 10. Summary table + Welch t-tests

In [ ]:
from scipy import stats

summary = {}
for name, arr in accs.items():
    a = np.array(arr)
    summary[name] = {"mean": float(a.mean()), "std": float(a.std()),
                      "n": int(a.size), "accs": a.tolist()}
    print(f"  {name:18s}  {a.mean():.4f} ± {a.std():.4f}   (n={a.size})")

print("\n=== Welch t-tests (paired) ===")
pairs = [("fractal", "protonets"), ("fractal", "pixel_knn"),
         ("fractal", "fractal_no_ctx"), ("protonets", "pixel_knn")]
tests = {}
for a, b in pairs:
    aa, bb = np.array(accs[a]), np.array(accs[b])
    t, p = stats.ttest_ind(aa, bb, equal_var=False)
    delta = aa.mean() - bb.mean()
    sig = "**" if p < 0.01 else ("*" if p < 0.05 else "(NS)")
    tests[f"{a}_vs_{b}"] = {"delta": float(delta), "welch_t": float(t), "welch_p": float(p)}
    print(f"  {a} vs {b}:  Δ={delta:+.4f}   t={t:+.2f}   p={p:.4f}   {sig}")

## Cell 11. Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar plot of means ± std
ax = axes[0]
names = ["pixel_knn", "fractal_no_ctx", "protonets", "fractal"]
labels = ["pixel-kNN", "FractalTrainer\n(no ctx)", "MLP-ProtoNets", "FractalTrainer\n(+context)"]
means = [summary[n]["mean"] for n in names]
stds  = [summary[n]["std"]  for n in names]
colors = ["gray", "lightblue", "steelblue", "darkorange"]
ax.bar(range(4), means, yerr=stds, capsize=5, color=colors)
ax.set_xticks(range(4)); ax.set_xticklabels(labels, fontsize=9)
ax.axhline(0.2, color="gray", ls=":", alpha=0.5, label="chance (0.2)")
ax.set_ylabel("Accuracy"); ax.set_ylim(0, 1.02)
ax.set_title(f"Omniglot {N_WAY}-way {K_SHOT}-shot ({N_TEST_EPISODES} episodes)")
ax.legend(); ax.grid(axis="y", alpha=0.3)

# Per-episode scatter: fractal vs protonets
ax = axes[1]
fa = np.array(accs["fractal"]); pa = np.array(accs["protonets"])
ax.scatter(pa, fa, alpha=0.6, color="darkorange")
lo, hi = 0, 1.05
ax.plot([lo, hi], [lo, hi], "--", color="gray", alpha=0.5, label="y=x")
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel("ProtoNets accuracy per episode")
ax.set_ylabel("FractalTrainer accuracy per episode")
wins = int((fa > pa).sum()); ties = int((fa == pa).sum()); losses = int((fa < pa).sum())
ax.set_title(f"Per-episode: FractalTrainer {wins}W / {ties}T / {losses}L  vs ProtoNets")
ax.legend(); ax.grid(alpha=0.3)

fig.tight_layout()
_stem = "sprint19b_omniglot" + ("_smoke" if SMOKE else "")
plt_path = f"/content/{_stem}.png"
fig.savefig(plt_path, dpi=120, bbox_inches="tight"); plt.show()
print(f"saved: {plt_path}")

## Cell 12. Save results (triple persistence)

In [ ]:
import json, shutil

payload = {
    "config": {
        "mode": "SMOKE" if SMOKE else "FULL",
        "n_way": N_WAY, "k_shot": K_SHOT, "n_query": N_QUERY,
        "n_seed_episodes": N_SEED_EPISODES,
        "n_test_episodes": N_TEST_EPISODES,
        "context_k": CONTEXT_K,
        "spawn_steps": SPAWN_STEPS,
        "protonet_iters": PROTONET_ITERS,
        "probe_size": PROBE_SIZE, "sig_dim": SIG_DIM,
        "device": str(DEVICE), "git_head": head,
    },
    "summary": summary,
    "tests": tests,
}
json_path = f"/content/{_stem}.json"
with open(json_path, "w") as fh:
    json.dump(payload, fh, indent=2, default=float)
print(f"1. Local: {json_path}")

if DRIVE_PATH:
    shutil.copy(json_path, DRIVE_PATH); shutil.copy(plt_path, DRIVE_PATH)
    print(f"2. Drive: {DRIVE_PATH}/")
else:
    print("2. Drive: skipped")

print("3. Inline below — survives notebook save.")

## Cell 13. Final report (inline)

In [ ]:
print("=" * 70)
print(f"SPRINT 19b — OMNIGLOT {N_WAY}-WAY {K_SHOT}-SHOT ({'SMOKE' if SMOKE else 'FULL'})")
print("=" * 70)
print(f"git HEAD:   {head}")
print(f"device:     {DEVICE}")
print(f"seed_eps:   {N_SEED_EPISODES}  test_eps: {N_TEST_EPISODES}  context_k: {CONTEXT_K}")
print(f"spawn_steps: {SPAWN_STEPS}  protonet_iters: {PROTONET_ITERS}")
print()
print(f"{'method':>22s}  {'mean':>7s}  {'std':>6s}  n")
print("-" * 50)
for n in ["pixel_knn", "fractal_no_ctx", "protonets", "fractal"]:
    s = summary[n]
    print(f"{n:>22s}  {s['mean']:.4f}  {s['std']:.4f}  {s['n']}")
print()
print("pairwise:")
for k, v in tests.items():
    sig = "**" if v['welch_p'] < 0.01 else ("*" if v['welch_p'] < 0.05 else "(NS)")
    print(f"  {k:>30s}:  Δ={v['delta']:+.4f}   t={v['welch_t']:+.2f}   p={v['welch_p']:.4f}  {sig}")

print()
# Pre-registered verdicts
proto = summary["protonets"]["mean"]; fract = summary["fractal"]["mean"]; knn = summary["pixel_knn"]["mean"]
delta_fp = fract - proto
print("Pre-registered verdicts:")
if abs(delta_fp) <= 0.03:
    print(f"  FractalTrainer vs ProtoNets: TIE  (|Δ|={abs(delta_fp):.3f} ≤ 0.03)")
elif delta_fp > 0.03:
    print(f"  FractalTrainer WINS over ProtoNets  (Δ=+{delta_fp:.3f})")
else:
    print(f"  FractalTrainer LOSES to ProtoNets  (Δ={delta_fp:.3f})")
if fract - knn > 0.2:
    print(f"  ✓ Fractal > pixel-kNN by {fract-knn:.3f}")
else:
    print(f"  ✗ Fractal only beats pixel-kNN by {fract-knn:.3f} (<0.2 pp gap, backbone weak)")

print()
print("Inline JSON payload (copy-paste fallback):")
print(json.dumps(payload, indent=2, default=float))

## Cell 14. Browser download

In [ ]:
try:
    from google.colab import files
    files.download(json_path); files.download(plt_path)
except Exception as e:
    print("download unavailable:", e)
